## Reshaping from WIDE to LONG via melt

We have seen how to explore our data. Focusing on the RELATIONSHIPS between continuous variables this week.

But, we have also continued to see how to important it is to GROUP BY or CONDITION ON categorical variables.

There is an important DATA MANIPULATION task that will further help us use GROUPING to visually explore our data!!!

## Import Modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

## Read data

In [ ]:
penguins = sns.load_dataset('penguins')

In [ ]:
penguins.info()

## Motivate why we need this operation

We know how to look at a marginal distribution for a SINGLE variable.

In [ ]:
sns.displot(data = penguins, x='flipper_length_mm', kind='hist', bins=11)

plt.show()

But..to visualize every continuous variable's histogram...we need to CALL a lot of functions!

We need to iterate over all the continuous columns!!!!

You might think we can use a for-loop! But you would be WRONG!!!!!!

Instead...we want to use a simple line of code to show ALL continuous variable MARGINAL distributions in one figure!

Pandas allows this to happen!

In [ ]:
penguins.hist()

plt.show()

But what if I don't want to use Pandas? What if I want to use Seaborn?

What happens if we use the same style of syntax for Seaborn as we did with Pandas?

In [ ]:
sns.displot( data = penguins, kind='hist', aspect=2 )

plt.show()

Let's use the similar syntax to make a BOXPLOT!

In [ ]:
sns.catplot(data = penguins, kind='box', aspect=2)

plt.show()

In [ ]:
sns.catplot(data = penguins, kind='box', col='species', aspect=2)

plt.show()

The SCALE is completely dominated by a single column (variable). That's why we could NOT see all of the histograms using the default Seaborn function!

Instead, we want Seaborn to create separate FACETS for each column just as Pandas did.

BUT...Seaborn PREFERS data in a DIFFERENT format from Pandas.

Pandas likes **WIDE FORMAT** data where each column is a VARIABLE.

Seaborn however prefers **LONG FORMAT** data...where we have a column that denotes the variables!!!!!

Let's execute RESHAPING from WIDE to LONG format to see what long format does!!!!!

## Long Format

Let's begin by focusing JUST on the numeric columns. Let's separate out.

In [ ]:
penguins_features = penguins.select_dtypes('number').copy()

In [ ]:
penguins_features

To STACK or GATHER all 4 columns in the CURRENT WIDE FORMAT into LONG FORMAT...we need to **MELT** the data.

RESHAPING from WIDE to LONG is accomplished with the `.melt()` method.

In [ ]:
penguins_features.melt()

In [ ]:
penguins_features.melt().variable.value_counts()

In [ ]:
penguins_features.shape[0]

In [ ]:
penguins_features.melt().shape

In [ ]:
344 * 4

In [ ]:
penguins_features.melt()

By default we LOSE the original `.index` attribute when we MELT. Meaning we lose which row the data originally came from!

We can KEEP the original `.index` by setting the `ignore_index` argument to False!

In [ ]:
penguins_features.melt(ignore_index=False)

In [ ]:
penguins_features.melt(ignore_index=False).loc[ 0 ]

In [ ]:
penguins.loc[0,:]

In [ ]:
penguins_features

I prefer to CONVERT the `.index` attribute into a REGULAR column and RENAME it to `rowid`.

In [ ]:
penguins_features.reset_index().\
rename(columns={'index': 'rowid'})

HOWEVER...we cannot melt with the default arguments! 

We do NOT want to GATHER or STACK the `rowid` column with the other variables!

Instead, we need to specify that the `rowid` column uniquely DEFINES the ROW ID!

In [ ]:
penguins_features.reset_index().\
rename(columns={'index': 'rowid'}).\
melt(id_vars=['rowid'])

In [ ]:
lf = penguins_features.reset_index().\
rename(columns={'index': 'rowid'}).\
melt(id_vars=['rowid'])

In [ ]:
lf

In [ ]:
lf.loc[ lf.rowid == 0, : ]

In [ ]:
penguins_features

The original `penguins` DataFrame had NON-NUMERIC columns!

In [ ]:
penguins

In [ ]:
penguins_objects = penguins.select_dtypes(['object', 'str']).copy()

In [ ]:
penguins_objects

Let's define a list that contains the `rowid` column and the 3 object column names!

This list will therefore hold ALL columns that uniquely define the ROW!!!!

In [ ]:
id_cols = ['rowid'] + penguins_objects.columns.to_list()

In [ ]:
id_cols

We can use `id_cols` to identify the columns to NOT gather or STACK when we call `.melt()` method on the original data set!

In [ ]:
penguins.reset_index().\
rename(columns={'index': 'rowid'}).\
melt(id_vars=id_cols)

To be very precise we can even identify the columns to GATHER via `value_vars` argument.

In [ ]:
penguins.reset_index().\
rename(columns={'index': 'rowid'}).\
melt(id_vars=id_cols, value_vars=penguins_features.columns)

Let's assign this to a NEW object.

In [ ]:
penguins_lf = penguins.reset_index().\
rename(columns={'index': 'rowid'}).\
melt(id_vars=id_cols, value_vars=penguins_features.columns)

In [ ]:
penguins_lf

## Plotting with Long Format

We can now associate the `variable` column with FACETS in Seaborn!!!!!!

So let's call the `sns.displot()` function again but this time with the LONG FORMAT data and assigning `col` to `variable`.

In [ ]:
sns.displot(data = penguins_lf, x='value', col='variable', kind='hist')

plt.show()

We need to force FREE or NOT SHARING x and y axis scales!

In [ ]:
sns.displot(data = penguins_lf, x='value', col='variable', kind='hist',
            facet_kws={'sharex': False, 'sharey': False})

plt.show()

One more change....by default the histograms want to use the common bins in all facets...

In [ ]:
sns.displot(data = penguins_lf, x='value', col='variable', kind='hist',
            facet_kws={'sharex': False, 'sharey': False},
            common_bins=False)

plt.show()

We can do the same thing with KDE plots...but remember common_norm must be FALSE!

In [ ]:
sns.displot(data = penguins_lf, x='value', col='variable', kind='kde',
            facet_kws={'sharex': False, 'sharey': False},
            common_norm=False)

plt.show()

Why is the LONG FORMAT so useful!??!?!

Because we can easily GROUP BY categorical variables!

We can group EACH facet by a categorical variable to study the CONDITIONAL DISTRIBUTION for each numeric column!

In [ ]:
sns.displot(data = penguins_lf, x='value', col='variable', kind='hist',
            hue='species',
            facet_kws={'sharex': False, 'sharey': False},
            common_bins=False)

plt.show()

In [ ]:
sns.displot(data = penguins_lf, x='value', col='variable', kind='hist',
            hue='species', row='sex',
            facet_kws={'sharex': False, 'sharey': False},
            common_bins=False)

plt.show()

We have reshaped...and we kept ALL non-numeric columns...we can FACET by other variables!

In [ ]:
sns.displot(data = penguins_lf, x='value', col='variable', kind='kde',
            row='species', hue='sex',
            facet_kws={'sharex': False, 'sharey': False},
            common_norm=False)

plt.show()

We can use OTHER types of PLOTS from the LONG FORMAT without changing anything!

We can show the CONDITIONAL distributions for each numeric column grouped by `species` as boxplots!

In [ ]:
sns.catplot(data = penguins_lf, x='species', y='value', col='variable', col_wrap=2,
            kind='box', hue='variable',
            sharey=False)

plt.show()

We can include an additional grouping via the box `hue`. So let's examine the differences between Male and Female as the `hue`.

In [ ]:
sns.catplot(data = penguins_lf, x='species', y='value', col='variable', col_wrap=2,
            hue='sex',
            kind='box',
            sharey=False)

plt.show()

## Summary

Wide format is intended for studying RELATIONSHIPS between variables.

In [ ]:
penguins

Long format is intended for GROUPING variables together!

In [ ]:
penguins_lf